In [1]:
from pathlib import Path
import pandas as pd
import re
import unicodedata

PROJECT_ROOT = Path(
    r"D:\DS Project\credit-portfolio-risk-modeling"
)

REFERENCE_DIR = PROJECT_ROOT / "docs" / "reference"

LAYOUT_FILE = (
    REFERENCE_DIR / "file_layout_july_2026.xlsx"
)

ORIG_HEADER_FILE = (
    REFERENCE_DIR / "origination_data_file_header.txt"
)

PERF_HEADER_FILE = (
    REFERENCE_DIR / "performance_data_file_header.txt"
)

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "data_audit"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert LAYOUT_FILE.exists(), "Data Dictionary not found"
assert ORIG_HEADER_FILE.exists(), "Origination header not found"
assert PERF_HEADER_FILE.exists(), "Performance header not found"

print("Reference files verified.")

Reference files verified.


In [3]:
orig_layout = pd.read_excel(
    LAYOUT_FILE,
    sheet_name="Origination Data File",
    header=1
)

perf_layout = pd.read_excel(
    LAYOUT_FILE,
    sheet_name="Monthly Performance Data File",
    header=1
)

# Keep only relevant columns
mapping_columns = [
    "Field Position",
    "Attribute Name",
    "Data Type & Format",
    "Max Length"
]

orig_layout = orig_layout[mapping_columns].dropna(
    subset=["Field Position"]
).copy()

perf_layout = perf_layout[mapping_columns].dropna(
    subset=["Field Position"]
).copy()

orig_layout["Field Position"] = (
    orig_layout["Field Position"].astype(int)
)

perf_layout["Field Position"] = (
    perf_layout["Field Position"].astype(int)
)

print("Origination fields:", len(orig_layout))
print("Performance fields:", len(perf_layout))

Origination fields: 31
Performance fields: 35


In [4]:
def to_snake_case(name):

    name = str(name)

    # Remove trademark symbols
    name = name.replace("®", "").replace("™", "")

    # Normalize Unicode
    name = unicodedata.normalize("NFKD", name)

    # Lowercase
    name = name.lower()

    # Replace special characters with spaces
    name = re.sub(r"[^a-z0-9]+", "_", name)

    # Remove leading/trailing underscores
    name = name.strip("_")

    return name


orig_layout["standard_name"] = (
    orig_layout["Attribute Name"].apply(to_snake_case)
)

perf_layout["standard_name"] = (
    perf_layout["Attribute Name"].apply(to_snake_case)
)

display(orig_layout.head())
display(perf_layout.head())

,Field Position,Attribute Name,Data Type & Format,Max Length,standard_name
0,1,Classic FICO®,Numeric,4,classic_fico
1,2,First Payment Date,Date,6,first_payment_date
2,3,First Time Homebuyer Indicator,Alpha,1,first_time_homebuyer_indicator
3,4,Maturity Date,Date,6,maturity_date
4,5,Metropolitan Statistical Area (MSA) Or Metropo...,Numeric,5,metropolitan_statistical_area_msa_or_metropoli...


,Field Position,Attribute Name,Data Type & Format,Max Length,standard_name
0,1,Loan Identifier,Alpha Numeric - PYYQnXXXXXXX,12,loan_identifier
1,2,Period,Date,6,period
2,3,Current Actual UPB,"Numeric - 12,2",12,current_actual_upb
3,4,Current Loan Delinquency Status,Alpha Numeric,3,current_loan_delinquency_status
4,5,Loan Age,Numeric,3,loan_age


In [5]:
def read_header_file(path):

    with open(path, "r", encoding="utf-8-sig") as f:
        line = f.readline().strip()

    return line.split("|")


orig_header = read_header_file(ORIG_HEADER_FILE)
perf_header = read_header_file(PERF_HEADER_FILE)

print("Origination header fields:", len(orig_header))
print("Performance header fields:", len(perf_header))

Origination header fields: 31
Performance header fields: 35


In [6]:
def normalize_for_comparison(name):

    name = str(name).upper()
    name = name.replace("®", "").replace("™", "")

    return re.sub(r"[^A-Z0-9]", "", name)


def compare_headers(layout, header):

    results = []

    for i, row in layout.reset_index(drop=True).iterrows():

        dictionary_name = row["Attribute Name"]
        header_name = header[i] if i < len(header) else None

        results.append({
            "position": i + 1,
            "dictionary_name": dictionary_name,
            "header_name": header_name,
            "match": (
                normalize_for_comparison(dictionary_name)
                == normalize_for_comparison(header_name)
                if header_name is not None else False
            )
        })

    return pd.DataFrame(results)


orig_comparison = compare_headers(
    orig_layout,
    orig_header
)

perf_comparison = compare_headers(
    perf_layout,
    perf_header
)

print(
    "Origination matches:",
    orig_comparison["match"].sum(),
    "/",
    len(orig_comparison)
)

print(
    "Performance matches:",
    perf_comparison["match"].sum(),
    "/",
    len(perf_comparison)
)

display(orig_comparison)
display(perf_comparison)

Origination matches: 31 / 31
Performance matches: 35 / 35


,position,dictionary_name,header_name,match
0,1,Classic FICO®,CLASSIC FICO,True
1,2,First Payment Date,FIRST PAYMENT DATE,True
2,3,First Time Homebuyer Indicator,FIRST TIME HOMEBUYER INDICATOR,True
3,4,Maturity Date,MATURITY DATE,True
4,5,Metropolitan Statistical Area (MSA) Or Metropo...,METROPOLITAN STATISTICAL AREA (MSA) OR METROPO...,True
5,6,Mortgage Insurance Percentage (MI %),MORTGAGE INSURANCE PERCENTAGE (MI %),True
6,7,Number of Units,NUMBER OF UNITS,True
7,8,Occupancy Status,OCCUPANCY STATUS,True
8,9,Original Combined Loan-to-Value (CLTV),ORIGINAL COMBINED LOAN-TO-VALUE (CLTV),True
9,10,Original Debt-to-Income (DTI) Ratio,ORIGINAL DEBT-TO-INCOME (DTI) RATIO,True


,position,dictionary_name,header_name,match
0,1,Loan Identifier,LOAN IDENTIFIER,True
1,2,Period,PERIOD,True
2,3,Current Actual UPB,CURRENT ACTUAL UPB,True
3,4,Current Loan Delinquency Status,CURRENT LOAN DELINQUENCY STATUS,True
4,5,Loan Age,LOAN AGE,True
5,6,Remaining Months to Legal Maturity,REMAINING MONTHS TO LEGAL MATURITY,True
6,7,Underwriting Defect and Major Servicing Defect...,UNDERWRITING DEFECT AND MAJOR SERVICING DEFECT...,True
7,8,Modification Flag,MODIFICATION FLAG,True
8,9,Zero Balance Code,ZERO BALANCE CODE,True
9,10,Zero Balance Effective Date,ZERO BALANCE EFFECTIVE DATE,True


In [7]:
orig_mapping = orig_layout.copy()
orig_mapping.insert(0, "dataset", "origination")

perf_mapping = perf_layout.copy()
perf_mapping.insert(0, "dataset", "performance")

full_mapping = pd.concat(
    [orig_mapping, perf_mapping],
    ignore_index=True
)

display(full_mapping)

,dataset,Field Position,Attribute Name,Data Type & Format,Max Length,standard_name
0,origination,1,Classic FICO®,Numeric,4,classic_fico
1,origination,2,First Payment Date,Date,6,first_payment_date
2,origination,3,First Time Homebuyer Indicator,Alpha,1,first_time_homebuyer_indicator
3,origination,4,Maturity Date,Date,6,maturity_date
4,origination,5,Metropolitan Statistical Area (MSA) Or Metropo...,Numeric,5,metropolitan_statistical_area_msa_or_metropoli...
...,...,...,...,...,...,...
61,performance,31,Current Period Modification Costs,"Numeric - 12,2",12,current_period_modification_costs
62,performance,32,Current Interest Bearing UPB,"Numeric - 12,2",12,current_interest_bearing_upb
63,performance,33,Mortgage Insurance Cancellation Indicator,Alpha,1,mortgage_insurance_cancellation_indicator
64,performance,34,Servicer Name,Alpha Numeric,60,servicer_name


In [8]:
full_mapping.to_csv(
    OUTPUT_DIR / "freddie_mac_field_mapping.csv",
    index=False,
    encoding="utf-8-sig"
)

orig_mapping.to_csv(
    OUTPUT_DIR / "origination_field_mapping.csv",
    index=False,
    encoding="utf-8-sig"
)

perf_mapping.to_csv(
    OUTPUT_DIR / "performance_field_mapping.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Field mappings exported successfully.")

Field mappings exported successfully.


In [9]:
def validate_mapping(df, dataset_name):

    print(f"\n{dataset_name.upper()}")

    checks = {
        "Correct field count": len(df) == (
            31 if dataset_name == "origination" else 35
        ),

        "Unique positions": (
            df["Field Position"].is_unique
        ),

        "Continuous positions": (
            sorted(df["Field Position"].tolist())
            == list(range(1, len(df) + 1))
        ),

        "Unique standard names": (
            df["standard_name"].is_unique
        ),

        "No empty standard names": (
            df["standard_name"].notna().all()
            and df["standard_name"].ne("").all()
        )
    }

    for check, passed in checks.items():
        print(f"{check}: {'PASS' if passed else 'FAIL'}")

    return all(checks.values())


orig_passed = validate_mapping(
    orig_mapping, "origination"
)

perf_passed = validate_mapping(
    perf_mapping, "performance"
)

assert orig_passed and perf_passed, (
    "Mapping integrity check failed."
)


ORIGINATION
Correct field count: PASS
Unique positions: PASS
Continuous positions: PASS
Unique standard names: PASS
No empty standard names: PASS

PERFORMANCE
Correct field count: PASS
Unique positions: PASS
Continuous positions: PASS
Unique standard names: PASS
No empty standard names: PASS


In [11]:
import sys

sys.path.append(str(PROJECT_ROOT))

from config.paths import get_vintage_file

orig_path = get_vintage_file("orig", 2012, 4)
perf_path = get_vintage_file("perf", 2012, 4)

print(orig_path)
print(perf_path)

E:\DS_project_backup\credit-risk-portfolio\data\orig_2012Q4.txt
E:\DS_project_backup\credit-risk-portfolio\data\perf_2012Q4.txt


In [13]:

# Cell 7 — Load 2012Q4 sample files

import pandas as pd

# Get file paths from config/paths.py
orig_path = get_vintage_file("orig", 2012, 4)
perf_path = get_vintage_file("perf", 2012, 4)

# Get standardized column names from Data Dictionary
orig_names = (
    orig_mapping
    .sort_values("Field Position")["standard_name"]
    .tolist()
)

perf_names = (
    perf_mapping
    .sort_values("Field Position")["standard_name"]
    .tolist()
)

# Read samples as strings to preserve raw values
orig_raw_sample = pd.read_csv(
    orig_path,
    sep="|",
    header=None,
    names=orig_names,
    dtype="string",
    nrows=10000,
    keep_default_na=False
)

perf_raw_sample = pd.read_csv(
    perf_path,
    sep="|",
    header=None,
    names=perf_names,
    dtype="string",
    nrows=10000,
    keep_default_na=False
)

print("Origination sample:", orig_raw_sample.shape)
print("Performance sample:", perf_raw_sample.shape)


Origination sample: (10000, 31)
Performance sample: (10000, 35)


In [14]:
def classify_data_type(value):

    value = str(value).strip().lower()

    if value.startswith("numeric"):
        return "numeric"

    if value.startswith("date"):
        return "date"

    if value.startswith("alpha numeric"):
        return "alpha_numeric"

    if value.startswith("alpha"):
        return "alpha"

    return "other"


orig_mapping["data_category"] = (
    orig_mapping["Data Type & Format"]
    .apply(classify_data_type)
)

perf_mapping["data_category"] = (
    perf_mapping["Data Type & Format"]
    .apply(classify_data_type)
)

print("Origination:")
display(
    orig_mapping["data_category"].value_counts()
)

print("Performance:")
display(
    perf_mapping["data_category"].value_counts()
)

Origination:


data_category
numeric          13
alpha            12
alpha_numeric     4
date              2
Name: count, dtype: int64

Performance:


data_category
numeric          22
alpha             6
date              4
alpha_numeric     3
Name: count, dtype: int64

In [15]:
def numeric_validation(df, mapping):

    results = []

    numeric_fields = mapping.loc[
        mapping["data_category"] == "numeric",
        "standard_name"
    ]

    for col in numeric_fields:

        values = df[col].astype("string").str.strip()

        nonblank = values.ne("")

        parsed = pd.to_numeric(
            values[nonblank],
            errors="coerce"
        )

        invalid_count = parsed.isna().sum()

        results.append({
            "field": col,
            "nonblank_count": int(nonblank.sum()),
            "valid_numeric": int(parsed.notna().sum()),
            "invalid_numeric": int(invalid_count),
            "invalid_rate_pct": round(
                invalid_count / max(nonblank.sum(), 1) * 100,
                3
            )
        })

    return pd.DataFrame(results)


orig_numeric_audit = numeric_validation(
    orig_raw_sample,
    orig_mapping
)

perf_numeric_audit = numeric_validation(
    perf_raw_sample,
    perf_mapping
)

display(orig_numeric_audit)
display(perf_numeric_audit)

,field,nonblank_count,valid_numeric,invalid_numeric,invalid_rate_pct
0,classic_fico,10000,10000,0,0.0
1,metropolitan_statistical_area_msa_or_metropoli...,6775,6775,0,0.0
2,mortgage_insurance_percentage_mi,10000,10000,0,0.0
3,number_of_units,10000,10000,0,0.0
4,original_combined_loan_to_value_cltv,10000,10000,0,0.0
5,original_debt_to_income_dti_ratio,10000,10000,0,0.0
6,original_upb,10000,10000,0,0.0
7,original_loan_to_value_ltv,10000,10000,0,0.0
8,original_interest_rate,10000,10000,0,0.0
9,original_loan_term,10000,10000,0,0.0


,field,nonblank_count,valid_numeric,invalid_numeric,invalid_rate_pct
0,current_actual_upb,10000,10000,0,0.0
1,loan_age,10000,10000,0,0.0
2,remaining_months_to_legal_maturity,10000,10000,0,0.0
3,zero_balance_code,83,83,0,0.0
4,current_interest_rate,10000,10000,0,0.0
5,current_non_interest_bearing_upb,10000,10000,0,0.0
6,mi_recoveries,0,0,0,0.0
7,net_sales_proceeds,0,0,0,0.0
8,non_mi_recoveries,0,0,0,0.0
9,total_expenses,0,0,0,0.0


In [16]:
def missing_value_audit(df):

    results = []

    for col in df.columns:

        blank_count = (
            df[col]
            .astype("string")
            .str.strip()
            .eq("")
            .sum()
        )

        results.append({
            "field": col,
            "blank_count": int(blank_count),
            "blank_rate_pct": round(
                blank_count / len(df) * 100,
                2
            )
        })

    return (
        pd.DataFrame(results)
        .sort_values("blank_rate_pct", ascending=False)
    )


orig_missing_audit = missing_value_audit(
    orig_raw_sample
)

perf_missing_audit = missing_value_audit(
    perf_raw_sample
)

display(orig_missing_audit.head(15))
display(perf_missing_audit.head(15))

,field,blank_count,blank_rate_pct
26,special_eligibility_program,10000,100.00
25,pre_harp_loan_sequence_number,8546,85.46
4,metropolitan_statistical_area_msa_or_metropoli...,3225,32.25
2,first_time_homebuyer_indicator,0,0.00
3,maturity_date,0,0.00
1,first_payment_date,0,0.00
0,classic_fico,0,0.00
7,occupancy_status,0,0.00
8,original_combined_loan_to_value_cltv,0,0.00
9,original_debt_to_income_dti_ratio,0,0.00


,field,blank_count,blank_rate_pct
18,maintenance_and_preservation_costs,10000,100.00
7,modification_flag,10000,100.00
6,underwriting_defect_and_major_servicing_defect...,10000,100.00
23,interest_rate_step_indicator,10000,100.00
21,actual_loss,10000,100.00
20,miscellaneous_expenses,10000,100.00
17,legal_costs,10000,100.00
15,non_mi_recoveries,10000,100.00
16,total_expenses,10000,100.00
13,mi_recoveries,10000,100.00


In [17]:
categorical_fields = [
    "first_time_homebuyer_indicator",
    "occupancy_status",
    "channel",
    "amortization_type",
    "property_state",
    "property_type",
    "loan_purpose",
    "current_loan_delinquency_status",
    "zero_balance_code",
    "modification_flag"
]

for col in categorical_fields:

    if col not in orig_raw_sample.columns:
        if col not in perf_raw_sample.columns:
            continue
        df = perf_raw_sample
    else:
        df = orig_raw_sample

    print(f"\n{col}")

    print(
        df[col]
        .value_counts(dropna=False)
        .head(20)
    )


first_time_homebuyer_indicator
first_time_homebuyer_indicator
N    9707
Y     293
Name: count, dtype: Int64

occupancy_status
occupancy_status
P    9094
I     506
S     400
Name: count, dtype: Int64

channel
channel
R    9298
B     609
C      93
Name: count, dtype: Int64

amortization_type
amortization_type
FRM    10000
Name: count, dtype: Int64

property_state
property_state
IL    1321
MI     803
KY     585
NY     542
IA     481
CA     470
WI     451
IN     436
MN     349
OH     281
MO     270
KS     260
CT     255
NC     255
MA     246
OR     235
HI     204
ME     178
NE     174
AZ     171
Name: count, dtype: Int64

property_type
property_type
SF    8479
PU     922
CO     503
MH      96
Name: count, dtype: Int64

loan_purpose
loan_purpose
N    5883
C    2789
P    1328
Name: count, dtype: Int64

current_loan_delinquency_status
current_loan_delinquency_status
00    9843
01      87
02      34
03      16
04       4
05       3
06       3
07       3
08       2
09       1
10       1
11    

In [18]:
numeric_fields_to_check = [
    "classic_fico",
    "original_upb",
    "original_loan_to_value_ltv",
    "original_combined_loan_to_value_cltv",
    "original_debt_to_income_dti_ratio",
    "original_interest_rate",
    "current_actual_upb",
    "current_loan_age",
    "remaining_months_to_legal_maturity"
]

for col in numeric_fields_to_check:

    if col not in orig_raw_sample.columns:
        if col not in perf_raw_sample.columns:
            continue
        df = perf_raw_sample
    else:
        df = orig_raw_sample

    values = pd.to_numeric(
        df[col].replace("", pd.NA),
        errors="coerce"
    )

    print(f"\n{col}")
    print(values.describe())


classic_fico
count       10000.0
mean       762.6065
std      101.682425
min           509.0
25%           737.0
50%           773.0
75%           794.0
max          9999.0
Name: classic_fico, dtype: Float64

original_upb
count         10000.0
mean         174391.1
std      96225.804654
min           16000.0
25%          101000.0
50%          154000.0
75%          227000.0
max          645000.0
Name: original_upb, dtype: Float64

original_loan_to_value_ltv
count      10000.0
mean       73.1341
std      24.108995
min           10.0
25%           60.0
50%           75.0
75%           80.0
max          269.0
Name: original_loan_to_value_ltv, dtype: Float64

original_combined_loan_to_value_cltv
count     10000.0
mean       74.303
std      24.49573
min          10.0
25%          61.0
50%          75.0
75%          80.0
max         356.0
Name: original_combined_loan_to_value_cltv, dtype: Float64

original_debt_to_income_dti_ratio
count       10000.0
mean       170.6563
std      341.821511
m

In [19]:
def zero_blank_audit(df, fields):

    results = []

    for col in fields:

        values = df[col].astype("string").str.strip()

        blank_count = values.eq("").sum()
        zero_count = values.isin(["0", "0.0", "0.00"]).sum()

        results.append({
            "field": col,
            "blank_count": int(blank_count),
            "zero_count": int(zero_count),
            "nonblank_nonzero": int(
                (values.ne("") & ~values.isin(
                    ["0", "0.0", "0.00"]
                )).sum()
            )
        })

    return pd.DataFrame(results)


performance_loss_fields = [
    "mi_recoveries",
    "net_sales_proceeds",
    "non_mi_recoveries",
    "total_expenses",
    "actual_loss"
]

display(
    zero_blank_audit(
        perf_raw_sample,
        performance_loss_fields
    )
)

,field,blank_count,zero_count,nonblank_nonzero
0,mi_recoveries,10000,0,0
1,net_sales_proceeds,10000,0,0
2,non_mi_recoveries,10000,0,0
3,total_expenses,10000,0,0
4,actual_loss,10000,0,0


In [20]:
special_values = {
    "classic_fico": ["9999"],
    "original_debt_to_income_dti_ratio": ["999"],
    "original_loan_to_value_ltv": ["999"],
    "original_combined_loan_to_value_cltv": ["999"],
    "mortgage_insurance_percentage_mi": ["999"],
    "number_of_units": ["99"],
}

special_records = []

for field, codes in special_values.items():

    values = orig_raw_sample[field].astype("string").str.strip()

    for code in codes:

        count = values.eq(code).sum()

        special_records.append({
            "field": field,
            "special_code": code,
            "count": int(count),
            "rate_pct": round(
                count / len(orig_raw_sample) * 100, 3
            )
        })

special_audit = pd.DataFrame(special_records)

display(special_audit)

,field,special_code,count,rate_pct
0,classic_fico,9999,1,0.01
1,original_debt_to_income_dti_ratio,999,1454,14.54
2,original_loan_to_value_ltv,999,0,0.00
3,original_combined_loan_to_value_cltv,999,0,0.00
4,mortgage_insurance_percentage_mi,999,0,0.00
5,number_of_units,99,0,0.00


In [21]:
harp_check = orig_raw_sample[
    [
        "harp_indicator",
        "original_loan_to_value_ltv",
        "original_combined_loan_to_value_cltv",
        "original_debt_to_income_dti_ratio"
    ]
].copy()

for col in harp_check.columns[1:]:
    harp_check[col] = pd.to_numeric(
        harp_check[col].replace("", pd.NA),
        errors="coerce"
    )

harp_check["harp_indicator"] = (
    harp_check["harp_indicator"]
    .astype("string")
    .str.strip()
)

display(
    harp_check.groupby("harp_indicator", dropna=False)
    .agg(
        loan_count=("original_loan_to_value_ltv", "size"),
        mean_ltv=("original_loan_to_value_ltv", "mean"),
        max_ltv=("original_loan_to_value_ltv", "max"),
        max_cltv=("original_combined_loan_to_value_cltv", "max")
    )
)

,loan_count,mean_ltv,max_ltv,max_cltv
harp_indicator,,,,
N,8546,67.912825,95,95
Y,1454,103.822558,269,356


In [22]:
performance_event_audit = pd.DataFrame({
    "zero_balance_code": (
        perf_raw_sample["zero_balance_code"]
        .value_counts(dropna=False)
    )
})

display(performance_event_audit)

# Check zero UPB and delinquency status
print(
    "Zero current UPB:",
    pd.to_numeric(
        perf_raw_sample["current_actual_upb"],
        errors="coerce"
    ).eq(0).sum()
)

print(
    "Nonblank Actual Loss:",
    perf_raw_sample["actual_loss"].str.strip().ne("").sum()
)

,zero_balance_code
zero_balance_code,
,9917
01,83


Zero current UPB: 83
Nonblank Actual Loss: 0


In [23]:
dti_harp = orig_raw_sample.copy()

dti_harp["dti_special"] = (
    dti_harp["original_debt_to_income_dti_ratio"]
    .astype("string")
    .str.strip()
    .eq("999")
)

dti_harp["harp_indicator"] = (
    dti_harp["harp_indicator"]
    .astype("string")
    .str.strip()
)

display(
    pd.crosstab(
        dti_harp["harp_indicator"],
        dti_harp["dti_special"],
        margins=True
    )
)

dti_special,False,True,All
harp_indicator,,,
N,8546,0,8546
Y,0,1454,1454
All,8546,1454,10000


In [24]:
perf_check = perf_raw_sample.copy()

perf_check["upb_numeric"] = pd.to_numeric(
    perf_check["current_actual_upb"],
    errors="coerce"
)

perf_check["zero_upb"] = (
    perf_check["upb_numeric"].eq(0)
)

perf_check["zero_balance_code"] = (
    perf_check["zero_balance_code"]
    .astype("string")
    .str.strip()
)

display(
    pd.crosstab(
        perf_check["zero_balance_code"],
        perf_check["zero_upb"],
        margins=True
    )
)

zero_upb,False,True,All
zero_balance_code,,,
,9917,0,9917
01,0,83,83
All,9917,83,10000


In [25]:
loss_check = perf_raw_sample.copy()

loss_check["actual_loss_numeric"] = pd.to_numeric(
    loss_check["actual_loss"].replace("", pd.NA),
    errors="coerce"
)

print(
    "Nonblank Actual Loss:",
    loss_check["actual_loss_numeric"].notna().sum()
)

print(
    "Nonzero Actual Loss:",
    loss_check["actual_loss_numeric"].ne(0).sum()
)

print(
    "Nonblank Net Sales Proceeds:",
    loss_check["net_sales_proceeds"].str.strip().ne("").sum()
)

print(
    "Nonblank Total Expenses:",
    loss_check["total_expenses"].str.strip().ne("").sum()
)

Nonblank Actual Loss: 0
Nonzero Actual Loss: 0
Nonblank Net Sales Proceeds: 0
Nonblank Total Expenses: 0


In [26]:
from pathlib import Path
import pandas as pd

special_code_rules = {
    "classic_fico": "9999",
    "original_debt_to_income_dti_ratio": "999",
    "original_loan_to_value_ltv": "999",
    "original_combined_loan_to_value_cltv": "999",
    "mortgage_insurance_percentage_mi": "999",
    "number_of_units": "99"
}

# Build column positions from Data Dictionary
orig_positions = (
    orig_mapping
    .set_index("standard_name")["Field Position"]
    .to_dict()
)

audit_results = []

for year in range(2006, 2013):

    for quarter in range(1, 5):

        file_path = get_vintage_file(
            "orig", year, quarter
        )

        selected_columns = sorted(
            orig_positions[col] - 1
            for col in special_code_rules
        )

        sample = pd.read_csv(
            file_path,
            sep="|",
            header=None,
            usecols=selected_columns,
            dtype="string",
            keep_default_na=False
        )

        for field, special_code in special_code_rules.items():

            col_position = orig_positions[field] - 1

            count = (
                sample.iloc[:, selected_columns.index(col_position)]
                .str.strip()
                .eq(special_code)
                .sum()
            )

            audit_results.append({
                "vintage": f"{year}Q{quarter}",
                "field": field,
                "special_code": special_code,
                "count": int(count),
                "total_rows": len(sample),
                "rate_pct": round(
                    count / len(sample) * 100, 3
                ) if len(sample) else 0
            })

cross_vintage_special_audit = pd.DataFrame(
    audit_results
)

display(cross_vintage_special_audit)

,vintage,field,special_code,count,total_rows,rate_pct
0,2006Q1,classic_fico,9999,244,298574,0.082
1,2006Q1,original_debt_to_income_dti_ratio,999,8240,298574,2.760
2,2006Q1,original_loan_to_value_ltv,999,33,298574,0.011
3,2006Q1,original_combined_loan_to_value_cltv,999,33,298574,0.011
4,2006Q1,mortgage_insurance_percentage_mi,999,1,298574,0.000
...,...,...,...,...,...,...
163,2012Q4,original_debt_to_income_dti_ratio,999,187423,592713,31.621
164,2012Q4,original_loan_to_value_ltv,999,8,592713,0.001
165,2012Q4,original_combined_loan_to_value_cltv,999,9,592713,0.002
166,2012Q4,mortgage_insurance_percentage_mi,999,0,592713,0.000


In [27]:
cross_vintage_special_audit.to_csv(
    OUTPUT_DIR / "cross_vintage_special_code_audit.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# Aggregate special code audit

special_summary = (
    cross_vintage_special_audit
    .groupby(["field", "special_code"], as_index=False)
    .agg(
        total_special_count=("count", "sum"),
        total_loans=("total_rows", "sum"),
        min_rate_pct=("rate_pct", "min"),
        max_rate_pct=("rate_pct", "max")
    )
)

special_summary["overall_rate_pct"] = (
    special_summary["total_special_count"]
    / special_summary["total_loans"] * 100
).round(3)

special_summary = special_summary.sort_values(
    "overall_rate_pct",
    ascending=False
)

display(special_summary)

,field,special_code,total_special_count,total_loans,min_rate_pct,max_rate_pct,overall_rate_pct
4,original_debt_to_income_dti_ratio,999,1974757,11128442,0.506,39.222,17.745
0,classic_fico,9999,3067,11128442,0.001,0.107,0.028
2,number_of_units,99,2053,11128442,0.000,0.350,0.018
3,original_combined_loan_to_value_cltv,999,477,11128442,0.001,0.012,0.004
5,original_loan_to_value_ltv,999,461,11128442,0.001,0.012,0.004
1,mortgage_insurance_percentage_mi,999,57,11128442,0.000,0.005,0.001


In [29]:
special_summary.to_csv(
    OUTPUT_DIR / "special_code_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# DTI special code trend

dti_trend = (
    cross_vintage_special_audit[
        cross_vintage_special_audit["field"]
        == "original_debt_to_income_dti_ratio"
    ]
    .copy()
)

dti_trend = dti_trend.sort_values("vintage")

display(
    dti_trend[
        ["vintage", "count", "total_rows", "rate_pct"]
    ]
)

,vintage,count,total_rows,rate_pct
1,2006Q1,8240,298574,2.760
7,2006Q2,6972,308208,2.262
13,2006Q3,5302,283975,1.867
19,2006Q4,5132,302787,1.695
25,2007Q1,5517,302331,1.825
31,2007Q2,7504,354055,2.119
37,2007Q3,7294,284980,2.559
43,2007Q4,8698,296826,2.930
49,2008Q1,10784,421635,2.558
55,2008Q2,8714,370982,2.349


In [ ]:
# Cross-vintage HARP and DTI validation

harp_dti_results = []

for year in range(2006, 2013):

    for quarter in range(1, 5):

        file_path = get_vintage_file(
            "orig", year, quarter
        )

        df = pd.read_csv(
            file_path,
            sep="|",
            header=None,
            usecols=[
                orig_positions["original_debt_to_income_dti_ratio"] - 1,
                orig_positions["harp_indicator"] - 1
            ],
            dtype="string",
            keep_default_na=False
        )

        df.columns = [
            "dti",
            "harp_indicator"
        ]

        df["dti_special"] = (
            df["dti"].str.strip() == "999"
        )

        df["harp_indicator"] = (
            df["harp_indicator"].str.strip()
        )

        table = pd.crosstab(
            df["harp_indicator"],
            df["dti_special"]
        )

        harp_dti_results.append({
            "vintage": f"{year}Q{quarter}",
            "harp_yes": int(
                (df["harp_indicator"] == "Y").sum()
            ),
            "harp_yes_dti_special": int(
                (
                    (df["harp_indicator"] == "Y")
                    & df["dti_special"]
                ).sum()
            ),
            "non_harp_dti_special": int(
                (
                    (df["harp_indicator"] != "Y")
                    & df["dti_special"]
                ).sum()
            ),
            "dti_special_total": int(
                df["dti_special"].sum()
            )
        })

harp_dti_audit = pd.DataFrame(harp_dti_results)

display(harp_dti_audit)

,vintage,harp_yes,harp_yes_dti_special,non_harp_dti_special,dti_special_total
0,2006Q1,0,0,8240,8240
1,2006Q2,0,0,6972,6972
2,2006Q3,0,0,5302,5302
3,2006Q4,0,0,5132,5132
4,2007Q1,0,0,5517,5517
5,2007Q2,0,0,7504,7504
6,2007Q3,0,0,7294,7294
7,2007Q4,0,0,8698,8698
8,2008Q1,0,0,10784,10784
9,2008Q2,1,1,8713,8714


In [ ]:
# HARP indicator value audit

harp_value_results = []

for year in range(2006, 2013):

    for quarter in range(1, 5):

        file_path = get_vintage_file(
            "orig", year, quarter
        )

        harp_position = (
            orig_positions["harp_indicator"] - 1
        )

        df = pd.read_csv(
            file_path,
            sep="|",
            header=None,
            usecols=[harp_position],
            dtype="string",
            keep_default_na=False
        )

        df.columns = ["harp_indicator"]

        counts = (
            df["harp_indicator"]
            .str.strip()
            .value_counts(dropna=False)
        )

        for code, count in counts.items():

            harp_value_results.append({
                "vintage": f"{year}Q{quarter}",
                "harp_code": code,
                "count": int(count)
            })

harp_value_audit = pd.DataFrame(
    harp_value_results
)

display(
    harp_value_audit
    .groupby("harp_code", as_index=False)["count"]
    .sum()
    .sort_values("count", ascending=False)
)

,harp_code,count
0,N,9241010
1,Y,1887432


In [ ]:
# HARP and DTI special-code rates

import numpy as np
import pandas as pd

harp_dti_rates = harp_dti_audit.copy()

# Ensure numeric data types
numeric_cols = [
    "harp_yes",
    "harp_yes_dti_special",
    "dti_special_total"
]

for col in numeric_cols:
    harp_dti_rates[col] = pd.to_numeric(
        harp_dti_rates[col],
        errors="coerce"
    )

# Percentage of HARP loans with DTI special code
harp_dti_rates["harp_dti_special_rate_pct"] = (
    harp_dti_rates["harp_yes_dti_special"]
    .div(harp_dti_rates["harp_yes"].replace(0, np.nan))
    .mul(100)
    .round(3)
)

# Percentage of DTI special-code loans that are HARP
harp_dti_rates["dti_special_harp_share_pct"] = (
    harp_dti_rates["harp_yes_dti_special"]
    .div(harp_dti_rates["dti_special_total"].replace(0, np.nan))
    .mul(100)
    .round(3)
)

display(harp_dti_rates)

,vintage,harp_yes,harp_yes_dti_special,non_harp_dti_special,dti_special_total,harp_dti_special_rate_pct,dti_special_harp_share_pct
0,2006Q1,0,0,8240,8240,NaN,0.000
1,2006Q2,0,0,6972,6972,NaN,0.000
2,2006Q3,0,0,5302,5302,NaN,0.000
3,2006Q4,0,0,5132,5132,NaN,0.000
4,2007Q1,0,0,5517,5517,NaN,0.000
5,2007Q2,0,0,7504,7504,NaN,0.000
6,2007Q3,0,0,7294,7294,NaN,0.000
7,2007Q4,0,0,8698,8698,NaN,0.000
8,2008Q1,0,0,10784,10784,NaN,0.000
9,2008Q2,1,1,8713,8714,100.0,0.011


In [ ]:
# Save final HARP-DTI audit outputs

harp_value_audit.to_csv(
    OUTPUT_DIR / "harp_indicator_value_audit.csv",
    index=False,
    encoding="utf-8-sig"
)

harp_dti_rates.to_csv(
    OUTPUT_DIR / "harp_dti_cross_vintage_analysis.csv",
    index=False,
    encoding="utf-8-sig"
)


HARP-DTI audit outputs saved successfully.


In [37]:
import shutil
from pathlib import Path

drives = ["D:\\", "E:\\"]

for drive in drives:

    usage = shutil.disk_usage(drive)

    print(f"\nDrive {drive}")
    print(f"Total: {usage.total / 1024**3:.2f} GB")
    print(f"Used: {usage.used / 1024**3:.2f} GB")
    print(f"Free: {usage.free / 1024**3:.2f} GB")


Drive D:\
Total: 102.35 GB
Used: 86.23 GB
Free: 16.12 GB

Drive E:\
Total: 102.35 GB
Used: 49.67 GB
Free: 52.68 GB


In [38]:
import duckdb

print("DuckDB version:", duckdb.__version__)

DuckDB version: 2.0.0


In [39]:
%whos DataFrame

Variable                      Type         Data/Info
----------------------------------------------------
cross_vintage_special_audit   DataFrame    Shape: (168, 6)
df                            DataFrame    Shape: (592713, 1)
dti_harp                      DataFrame    Shape: (10000, 32)
dti_trend                     DataFrame    Shape: (28, 6)
full_mapping                  DataFrame    Shape: (66, 6)
harp_check                    DataFrame    Shape: (10000, 4)
harp_dti_audit                DataFrame    Shape: (28, 5)
harp_dti_rates                DataFrame    Shape: (28, 7)
harp_value_audit              DataFrame    Shape: (45, 3)
loss_check                    DataFrame    Shape: (10000, 36)
orig_comparison               DataFrame    Shape: (31, 4)
orig_layout                   DataFrame    Shape: (31, 5)
orig_mapping                  DataFrame    Shape: (31, 7)
orig_missing_audit            DataFrame    Shape: (31, 3)
orig_numeric_audit            DataFrame    Shape: (13, 5)
orig_raw

In [40]:
print(full_mapping.shape)
print(full_mapping.columns.tolist())

(66, 6)
['dataset', 'Field Position', 'Attribute Name', 'Data Type & Format', 'Max Length', 'standard_name']


In [42]:
import importlib
import config.paths as project_paths

importlib.reload(project_paths)

print("AUDIT_DIR:", project_paths.AUDIT_DIR)
print("Exists:", project_paths.AUDIT_DIR.exists())

print(
    "2012Q4 Origination:",
    project_paths.get_vintage_file("orig", 2012, 4)
)

AUDIT_DIR: E:\DS_project_backup\credit-risk-portfolio\processed\audit
Exists: True
2012Q4 Origination: E:\DS_project_backup\credit-risk-portfolio\data\orig_2012Q4.txt


In [43]:
# Save Origination and Performance field mappings

orig_mapping.to_csv(
    project_paths.AUDIT_DIR / "origination_mapping.csv",
    index=False,
    encoding="utf-8-sig"
)

perf_mapping.to_csv(
    project_paths.AUDIT_DIR / "performance_mapping.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Mapping files saved successfully.")

print("Origination:", 
      project_paths.AUDIT_DIR / "origination_mapping.csv")

print("Performance:", 
      project_paths.AUDIT_DIR / "performance_mapping.csv")

Mapping files saved successfully.
Origination: E:\DS_project_backup\credit-risk-portfolio\processed\audit\origination_mapping.csv
Performance: E:\DS_project_backup\credit-risk-portfolio\processed\audit\performance_mapping.csv
